# Handwritten OCR Playbook

This notebook helps you quickly test handwritten text recognition on local images and public datasets.

It uses **TrOCR** (Transformer OCR), which generally works better on handwriting than classic OCR tools.


## 1) Install dependencies (run once)


In [ ]:
# If needed, uncomment:
# !pip install -q transformers torch pillow matplotlib datasets


## 2) Load a pretrained handwriting model

You can switch between:
- `microsoft/trocr-base-handwritten` (lighter)
- `microsoft/trocr-large-handwritten` (usually better quality)


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch

MODEL_NAME = "microsoft/trocr-base-handwritten"
device = "cuda" if torch.cuda.is_available() else "cpu"

processor = TrOCRProcessor.from_pretrained(MODEL_NAME)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME).to(device)
print(f"Loaded {MODEL_NAME} on {device}")


## 3) OCR helper for local files


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def recognize_handwriting(image_path: str, max_new_tokens: int = 128):
    image = Image.open(image_path).convert("RGB")
    pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

    generated_ids = model.generate(pixel_values, max_new_tokens=max_new_tokens)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    plt.figure(figsize=(12, 4))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"Prediction: {text}")
    plt.show()

    return text


## 4) Test on your own handwritten image

Put any image in your project (for example `samples/my_note.jpg`) and run:


In [ ]:
# text = recognize_handwriting("samples/my_note.jpg")
# print(text)


## 5) Optional: quick test on a public handwriting dataset sample

This downloads one IAM sample via Hugging Face Datasets.


In [ ]:
# Requires internet
from datasets import load_dataset

ds = load_dataset("Teklia/IAM-line", split="train[:1]")
sample = ds[0]
image = sample["image"]
gt = sample.get("text", "<no ground truth field>")

# Save temp image for helper
tmp_path = "/tmp/iam_sample.png"
image.save(tmp_path)
pred = recognize_handwriting(tmp_path)

print("Ground truth:", gt)
print("Prediction  :", pred)


## 6) Useful handwriting OCR libraries/datasets

### Libraries
- **Transformers + TrOCR** (`microsoft/trocr-*-handwritten`)
- **PaddleOCR** (has handwriting support in some models)
- **EasyOCR** (quick baseline, weaker on difficult handwriting)
- **Tesseract** (best as baseline; handwriting quality often limited)

### Datasets
- **IAM Handwriting Database** (lines/words, classic benchmark)
- **CVL Database** (offline handwritten text)
- **Bentham** (historical manuscripts)
- **RIMES** (French handwritten docs)

For quickest experiments, use IAM-compatible samples + TrOCR.
